# Prompt construction check

This notebook imports the production prompt-construction helpers, inspects every `PromptRules` section, and reproduces the full placeholder substitution performed before `_ask_llm()` calls the model. It does not make an API call or invoke the fallback industry classifier for the mapped sample ticker.

In [ ]:
from pathlib import Path
import json
import sys
import re

# Jupyter normally starts this notebook inside notebooks/. Add the repo root
# so the top-level predict.py module can be imported.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from prompt_construction import *

## 1. Confirm the runtime files resolve

In [ ]:
runtime_paths = {
    "prompt": PROMPT_PATH,
    "global_playbook": GLOBAL_PATH,
    "industry_playbooks": INDUSTRY_PATH,
    "mappings_path": MAPPINGS_PATH,
    "dossier_path": DOSSIER_PATH
}

for name, path in runtime_paths.items():
    print(f"{name:20} exists={path.exists()}  path={path}")
    assert path.exists(), f"Missing runtime file: {path}"

## 2. Inspect available industries and choose one

In [ ]:
global_playbook = load_yaml(GLOBAL_PATH)
industry_playbooks = load_yaml(INDUSTRY_PATH)

reserved_keys = {"meta", "quarter_calibration", "sources"}
available_industries = [
    key for key, value in industry_playbooks.items()
    if key not in reserved_keys and isinstance(value, dict)
]

print("Global rules:", len(global_playbook.get("observations", [])))
print("Quarter calibration rules:", len(industry_playbooks.get("quarter_calibration", [])))
print("Industries:", available_industries)

TICKER = "CRC"
INDUSTRY = format_industry_tag(get_industry(TICKER))
assert INDUSTRY in available_industries

## 3. Load exactly the blocks used by the prompt

In [ ]:
dossier_text = get_dossier(TICKER)
has_valid_dossier = is_valid_dossier(dossier_text)
rules = load_prompt_rules(INDUSTRY, include_dossier_rule=has_valid_dossier)

print("\nGLOBAL RULES (first 4,000 characters)\n")
print(rules.global_rules[:4000])
print("\nINDUSTRY RULES (first 4,000 characters)\n")
print(rules.industry_rules[:4000])
print("\nVALID DOSSIER:", has_valid_dossier)

calibration_ids = [rule["id"] for rule in industry_playbooks.get("quarter_calibration", [])]
assert calibration_ids and all(rule_id in rules.industry_rules for rule_id in calibration_ids)

other_industry = next(name for name in available_industries if name != INDUSTRY)
other_rule_ids = [rule["id"] for rule in industry_playbooks[other_industry].get("rules", [])]
if other_rule_ids:
    assert other_rule_ids[0] not in rules.industry_rules, f"Rules from {other_industry} leaked into the prompt"

## 4. Reproduce `_ask_llm()` prompt construction

Edit the mapped sample ticker or summary below to exercise other events. This cell manually applies every current template placeholder, then compares its output with `construct_prompt()`.

In [ ]:
sample_summary = {
    "summary": [
        "Revenue exceeded consensus expectations.",
        "Management raised full-year guidance.",
        "Capital expenditure guidance also increased.",
    ]
}
ticker = TICKER
event_type = "EARNINGS_RELEASE"
dossier = dossier_text if has_valid_dossier else NO_CACHED_DOSSIER

summary_value = sample_summary.get("summary")
if isinstance(summary_value, list):
    summary_text = "\n".join(f"- {bullet}" for bullet in summary_value)
elif summary_value:
    summary_text = str(summary_value)
else:
    summary_text = json.dumps(sample_summary, ensure_ascii=False)
summary_text = summary_text[:8000]

prompt_template = PROMPT_PATH.read_text(encoding="utf-8")
prompt_template = re.sub(
    r"\A\s*<!--.*?-->\s*",
    "",
    prompt_template,
    count=1,
    flags=re.DOTALL,
)

user_prompt = (
    prompt_template
    .replace("{summary_text}", summary_text)
    .replace("{global_rules}", rules.global_rules)
    .replace("{industry_rules}", rules.industry_rules)
    .replace("{dossier}", dossier)
)

production_prompt = construct_prompt(summary_text, ticker)
assert user_prompt == production_prompt
print(user_prompt)

## 5. Validate the finished prompt

In [ ]:
expected_placeholders = [
    "{summary_text}",
    "{global_rules}",
    "{industry_rules}",
    "{dossier}",
]
unresolved = [item for item in expected_placeholders if item in user_prompt]

print(f"Final prompt length: {len(user_prompt):,} characters")